In [10]:
import numpy as np
import matplotlib.pyplot as plt
import time
import RNA

 # Archive

Pour boltzmann : 

In [11]:
def calculate_partition_function(unpaired_probs, paired_probs, beta, m=3):
    """
    Computes the partition function Z[i,j] for an RNA alignment.
    Based on the recursion: Z[i,j] = Z[i,j-1]*exp(beta*UNP(j)) + sum(...) [cite: 292-295]
    """
    n = len(unpaired_probs)
    # Z matrix including an extra index for Z[i, i-1] = 1.0 cases
    Z = np.zeros((n + 1, n + 1), dtype=np.float64)
    
    # Initialization [cite: 293]
    for i in range(n):
        Z[i, i] = np.exp(beta * unpaired_probs[i])
        Z[i, i-1] = 1.0
    if n > 0: Z[n, n-1] = 1.0

    # DP Recursion [cite: 292]
    for length in range(1, n):
        for i in range(n - length):
            j = i + length
            
            # Case 1: j is unpaired
            # Z[i, j] = Z[i, j-1] * exp(beta * UNP(j))
            z_val = Z[i, j-1] * np.exp(beta * unpaired_probs[j])
            
            # Case 2: j is paired with some k [cite: 295]
            # sum over k: Z[i, k-1] * exp(beta * PAIR(k,j)) * Z[k+1, j-1]
            for k in range(i, j - m):
                # Handle the k=i case where Z[i, k-1] is Z[i, i-1] = 1.0
                term_left = Z[i, k-1] if k > i else 1.0
                z_val += term_left * np.exp(beta * paired_probs[k, j]) * Z[k+1, j-1]
                
            Z[i, j] = z_val
            
    return Z

def stochastic_traceback(i, j, Z, unpaired_probs, paired_probs, beta, structure, m=3):
    """
    Produces a random substructure for indices i..j based on partition function Z [cite: 300-301].
    """
    if j < i:
        return

    # Select a random float for selecting the case [cite: 304]
    x = np.random.uniform(0, Z[i, j])
    
    # Check Case: j is unpaired [cite: 306]
    z_unpaired = (Z[i, j-1] if j > i else 1.0) * np.exp(beta * unpaired_probs[j])
    
    if x < z_unpaired:
        stochastic_traceback(i, j-1, Z, unpaired_probs, paired_probs, beta, structure, m)
        return

    # Check Case: j is paired with k [cite: 309]
    cumulative_z = z_unpaired
    for k in range(i, j - m):
        term_left = Z[i, k-1] if k > i else 1.0
        z_paired = term_left * np.exp(beta * paired_probs[k, j]) * Z[k+1, j-1]
        cumulative_z += z_paired
        
        if x < cumulative_z:
            structure[k] = "("
            structure[j] = ")"
            # Recursively sample the two independent sub-problems [cite: 314]
            stochastic_traceback(i, k-1, Z, unpaired_probs, paired_probs, beta, structure, m)
            stochastic_traceback(k+1, j-1, Z, unpaired_probs, paired_probs, beta, structure, m)
            return

def sample_structures(unpaired_probs, paired_probs, beta, num_samples=1000, m=3):
    """Generates a list of sampled RnaMolecule structures."""
    n = len(unpaired_probs)
    Z = calculate_partition_function(unpaired_probs, paired_probs, beta, m)
    
    samples = []
    for _ in range(num_samples):
        struct_list = ["."] * n
        stochastic_traceback(0, n - 1, Z, unpaired_probs, paired_probs, beta, struct_list, m)
        samples.append("".join(struct_list))
    return samples

In [12]:
def parse_RNA_structure(dbstring):
    base_pairs = []
    stack = []
    
    for index, char in enumerate(dbstring):
        position = index + 1  
        
        if char == '(':
            stack.append(position)
        elif char == ')':
            if not stack:
                raise ValueError(f"Error at position {position} : No matching opening parenthesis")
            opening_pos = stack.pop()
            base_pairs.append((opening_pos, position))
        elif char == '.':
            continue
        else:
            raise ValueError(f"Incompatible character '{char}' detected at the {position}.")
            
    if stack:
        raise ValueError(f"Error : {len(stack)} No closed parenthesis (positions : {stack}).")
        
    return sorted(base_pairs)

In [13]:
def seq_without_gap(alignment_seq):
    """
    Extract the sequence without gaps and create a mapping from the pure sequence indices to the alignment indices.
    Returns:
    - seq_pure: the string without '-' characters
    - mapping: a dictionary { pure_index (1-based) : alignment_index (0-based) }
    """
    seq_pure = ""
    mapping = {}
    current_pure_idx = 1
    
    for i, char in enumerate(alignment_seq):
        if char != '-':
            seq_pure += char
            mapping[current_pure_idx] = i
            current_pure_idx += 1
            
    return seq_pure, mapping

In [14]:
def plot_probability_heatmap(samples):
    """Estimates and visualizes base pair probabilities from samples."""
    n = len(samples[0])
    prob_matrix = np.zeros((n, n))
    
    for s in samples:
        pairs = parse_RNA_structure(s) # Using your existing parse/pair function
        for i, j in pairs:
            # Note: handle 1-based vs 0-based indexing carefully
            prob_matrix[i-1, j-1] += 1
            
    prob_matrix /= len(samples)
    
    plt.imshow(prob_matrix, cmap='hot_r')
    plt.title(f"Estimated Base Pair Probabilities ({len(samples)} samples)")
    plt.colorbar(label="Probability")
    plt.show()

In [15]:
def plot_dot_plot(samples, sequence_name="RNA Sequence"):
    """
    Visualizes base pair probabilities using imshow to create a Dot Plot.
    
    Args:
        samples (list): List of dot-bracket strings from stochastic sampling.
        sequence_name (str): Name of the sequence for the plot title.
    """
    n = len(samples[0])
    # Initialize a matrix to store pair frequencies
    prob_matrix = np.zeros((n, n))
    
    # 1. Count base pair occurrences in samples
    for structure in samples:
        # Use your existing parse function to get pairs
        pairs = parse_RNA_structure(structure)
        for i, j in pairs:
            # Convert 1-based indexing from parse_RNA_structure to 0-based for numpy
            prob_matrix[i-1, j-1] += 1
            # Dot plots are often symmetric
            prob_matrix[j-1, i-1] += 1
            
    # 2. Normalize to get probabilities (0.0 to 1.0)
    prob_matrix /= len(samples)
    
    # 3. Create the plot using imshow
    plt.figure(figsize=(8, 8))
    # 'hot_r' (reversed hot) shows high probability as dark/red and low as white
    plt.imshow(prob_matrix, cmap='hot_r', interpolation='nearest')
    
    plt.title(f"Dot Plot: Base Pair Probabilities ({sequence_name})\nEstimated from {len(samples)} samples")
    plt.xlabel("Sequence Position")
    plt.ylabel("Sequence Position")
    plt.colorbar(label="Probability")

# --- Exemple d'utilisation ---
alignment_test = ["GGAGGAUUAGCUCAGCUGGGAGAGCAUCUGCCUUACAAGCAGAGGG-----------UCGGCGGUUCGAGCCCGUCAUCCUCC",
"GCCUUCCUAGCUCAG-UGGUAGAGCGCACGGCUUUUAACCGUGUGG-----------UCGUGGGUUCGAUCCCCACGGAAGGC",
"GCCUUUAUAGCUUAG-UGGUAAAGCGAUAAACUGAAGAUUUAUUUA-----------CAUGUAGUUCGAUUCUCAUUAAGGGC",
"GCGGAUAUAACUUAGGGGUUAAAGUUGCAGAUUGUGGCUCUGAAAA------------CACGGGUUCGAAUCCCGUUAUUCGC",
"GGAAAAUU-GAUCAUCGGCAAGAUAAGUUAUUUACUAAAUAAUAGGAUUUAAUAACCUGGUGAGUUCGAAUCUCACAUUUUCC"
]
unpaired_probs, paired_probs = get_mea_probabilities_with_conservation_score(alignment_test, l_factor=1.0, alpha=0.5)
beta = 1.0
samples = sample_structures(unpaired_probs, paired_probs, beta, num_samples=1000, m=2)
plot_dot_plot(samples, sequence_name="Test Alignment")

NameError: name 'get_mea_probabilities_with_conservation_score' is not defined

MEA :

In [ ]:

def get_mea_probabilities_with_mapping(alignments, l_factor):
    """
    Get the MEA probabilities for unpaired and paired bases with mapping from pure sequence to alignment columns.
    """
    n_cols = len(alignments[0])
    unpaired_acc = np.zeros(n_cols)
    paired_acc = np.zeros((n_cols, n_cols))
    
    for seq in alignments:
        # APPEL DE TA NOUVELLE FONCTION
        seq_pure, mapping = seq_without_gap(seq)
        n = len(seq_pure)
        
        # Calcul ViennaRNA sur la séquence réelle
        fc = RNA.fold_compound(seq_pure)
        fc.pf()
        
        # Initialisation locale pour cette séquence
        current_seq_paired = np.zeros((n_cols, n_cols))
        
        # Re-mapping des probabilités de paires
        bpp = fc.bpp()

        for i_pure in range(1, n + 1):
            for j_pure in range(i_pure + 1, n + 1):
                prob = bpp[i_pure][j_pure]
                if prob > 0:
                    col_i = mapping[i_pure]
                    col_j = mapping[j_pure]
                    current_seq_paired[col_i, col_j] = prob
        
        # Calcul des probabilités Unpaired (p_u)
        current_seq_unpaired = np.ones(n_cols)
        for i in range(n_cols):
            # Somme des probas où i est impliqué (en tant que premier ou deuxième membre)
            sum_p_ij = np.sum(current_seq_paired[i, :]) + np.sum(current_seq_paired[:, i])
            
            # Si c'est un gap, sum_p_ij sera 0, donc p_u reste 1.0 (Logique !)
            current_seq_unpaired[i] = max(0, 1.0 - sum_p_ij)

        # Accumulation pour l'alignement
        unpaired_acc += current_seq_unpaired
        paired_acc += (2 * l_factor * current_seq_paired)
        
    return unpaired_acc, paired_acc

def compute_conservation_score(alignments):
    n_cols = len(alignments[0])
    n_seqs = len(alignments)
    cons_matrix = np.zeros((n_cols, n_cols))
    
    valid_pairs = {'AU', 'UA', 'GC', 'CG', 'GU', 'UG'}
    
    for i in range(n_cols):
        for j in range(i + 4, n_cols):
            bonus = 0
            penalty = 0
            for seq in alignments:
                b1, b2 = seq[i].upper(), seq[j].upper()
                if b1 == '-' or b2 == '-': # Gap pénalisé
                    penalty += 1
                elif (b1 + b2) in valid_pairs:
                    bonus += 1
                else: # Paire impossible (ex: A-G) pénalisée
                    penalty += 1
            
            # On combine bonus et pénalité (on peut ajuster le poids de la pénalité)
            cons_matrix[i, j] = (bonus - penalty) / n_seqs
            
    return cons_matrix

def get_mea_probabilities_with_conservation_score(alignments, l_factor, alpha):
    """
    Combine les probabilités MEA avec le score de conservation.
    alpha : poids accordé au signal évolutif (conservation).
    """
    # 1. Calcul des probabilités de base (ton code précédent)
    unpaired_acc, paired_acc = get_mea_probabilities_with_mapping(alignments, l_factor)
    
    # 2. Calcul du score de conservation
    cons_score = compute_conservation_score(alignments)
    
    # 3. Fusion : PAIR(i,j) = Probabilités + (alpha * Conservation) 
    # On ajoute le signal de conservation à la matrice des paires
    paired_acc_final = paired_acc + (alpha * cons_score)
    
    return unpaired_acc, paired_acc_final

def mea_structure(alignments, unpaired_probs, paired_probs, min_dist):
    """ Calcule la structure MEA à partir des alignements en utilisant les probabilités de paires et non-paires avec mapping. """
    n_cols = len(alignments[0])
    
    # Matrice F pour la DP
    F = np.zeros((n_cols, n_cols))
    
    for k in range(1, n_cols):
        for i in range(n_cols - k):
            j = i + k
            
            # Options : unpaired i, unpaired j, paired (i,j), or bifurcation
            opts = [
                unpaired_probs[i] + F[i+1, j],
                unpaired_probs[j] + F[i, j-1]
            ]
            
            if k > min_dist:
                opts.append(paired_probs[i, j] + F[i+1, j-1])
            
            # Bifurcation
            for m in range(i, j):
                opts.append(F[i, m] + F[m+1, j])
                
            F[i, j] = max(opts)

    # Traceback
    consensus_structure = ["."] * n_cols

    def tb(i, j):
        if i >= j: return
        if F[i, j] == unpaired_probs[i] + F[i+1, j]:
            tb(i+1, j)
        elif F[i, j] == unpaired_probs[j] + F[i, j-1]:
            tb(i, j-1)
        elif i + min_dist < j and F[i, j] == paired_probs[i, j] + F[i+1, j-1]:
            consensus_structure[i], consensus_structure[j] = "(", ")"
            tb(i+1, j-1)
        else:
            for m in range(i, j):
                if F[i, j] == F[i, m] + F[m+1, j]:
                    tb(i, m)
                    tb(m+1, j)
                    break

    tb(0, n_cols - 1)
    return "".join(consensus_structure)

MEA Sparsified:

In [ ]:

def mea_structure_sparsified(unpaired_probs, paired_probs, m):
    n = len(unpaired_probs)
    F = np.zeros(n + 1) # On peut optimiser en 1D ou 2D selon la variante
    # Ici, implémentation inspirée du document source page 16
    F = np.zeros((n, n))
    
    for i in range(n):
        F[i, i] = unpaired_probs[i] # Init [cite: 188, 193]

    for j in range(n):
        candidates = [] # Liste des i qui font une bonne paire avec j [cite: 243]
        for i in reversed(range(j)):
            # Cas 1 : j est non apparié [cite: 246]
            f_score = F[i, j-1] + unpaired_probs[j]
            
            # Cas 2 : j est apparié avec un k précédent (Sparsification) [cite: 250]
            for k, c_score_kj in candidates:
                if i < k:
                    f_score = max(f_score, F[i, k-1] + c_score_kj)
            
            # Cas 3 : On teste si (i, j) lui-même est un bon candidat [cite: 252]
            if j - i > m:
                # Score de la paire (i, j) + structure interne
                c_ij = paired_probs[i, j] + F[i+1, j-1]
                if c_ij > f_score:
                    f_score = c_ij
                    candidates.append((i, c_ij)) # Ajout aux candidats [cite: 255]
            
            F[i, j] = f_score
    
    # Traceback
    consensus_structure = ["."] * n

    def tb(i, j):
        if i >= j: return
        if F[i, j] == unpaired_probs[i] + F[i+1, j]:
            tb(i+1, j)
        elif F[i, j] == unpaired_probs[j] + F[i, j-1]:
            tb(i, j-1)
        elif i + m < j and F[i, j] == paired_probs[i, j] + F[i+1, j-1]:
            consensus_structure[i], consensus_structure[j] = "(", ")"
            tb(i+1, j-1)
        else:
            for k in range(i, j):
                if F[i, j] == F[i, k] + F[k+1, j]:
                    tb(i, k)
                    tb(k+1, j)
                    break

    tb(0, n - 1)
    return "".join(consensus_structure)

In [ ]:
alignment_test = ["CGCAAA-GCG", "CGGAAAACCG"]

print("--- Verification of the Mapping ---")
for i, s in enumerate(alignment_test):
    pure, mapping = seq_without_gap(s)
    print(f"Original Sequence {i}   : {s}")
    print(f"Pure Sequence     {i}   : {pure}")
    print(f"Mapping (Pure -> Aln) : {mapping}\n")

print("--- MEA ---")
start = time.time()
unpairedprobs, paired_probs = get_mea_probabilities_with_conservation_score(alignment_test, l_factor=1.0, alpha=0.5)
struct_mea = mea_structure(alignment_test,unpairedprobs, paired_probs, min_dist=2)
end = time.time()
print(f"Process time of MEA : {end - start:.4f} secondes")
print(f"Final MEA Structure : {struct_mea}")


start = time.time()
struct_mea_sparsified = mea_structure_sparsified(*get_mea_probabilities_with_conservation_score(alignment_test, l_factor=1.0, alpha = 0.5), m=2)
end = time.time()
print(f"Process time of Sparsified MEA : {end - start:.4f} secondes")
print(f"Final Sparsified MEA Structure : {struct_mea_sparsified}")

Boltzmann un peu réarrangé: 


In [ ]:
def calculate_partition_function_stable(unpaired_probs, paired_probs, beta, m=3):
    """Computes the partition function with a scaling factor to prevent overflow."""
    n = len(unpaired_probs)
    Z = np.zeros((n + 1, n + 1), dtype=np.float64)
    
    avg_score = (np.mean(unpaired_probs) + np.mean(paired_probs)) 
    scale = np.exp(beta * avg_score * 1.1)

    # Initialization
    for i in range(n):
        Z[i, i] = np.exp(beta * unpaired_probs[i]) / scale
        Z[i, i-1] = 1.0
    if n > 0: Z[n, n-1] = 1.0

    # DP Recursion with scaling
    for length in range(1, n):
        for i in range(n - length):
            j = i + length
            # Scale unpaired case
            z_val = (Z[i, j-1] * np.exp(beta * unpaired_probs[j])) / scale
            
            # Scale paired cases
            for k in range(i, j - m):
                term_left = Z[i, k-1] if k > i else 1.0
                # Double-scaling factor for the two joined sub-problems
                z_val += (term_left * np.exp(beta * paired_probs[k, j]) * Z[k+1, j-1]) / (scale**2)
                
            Z[i, j] = z_val
            
    return Z, scale

def stochastic_traceback_stable(i, j, Z, unpaired_probs, paired_probs, beta, scale, structure, m=3):
    """Produces a random substructure using the scaled partition function."""
    if j < i:
        return
    
    x = np.random.uniform(0, Z[i, j])

    z_unpaired = (Z[i, j-1] if j > i else 1.0) * np.exp(beta * unpaired_probs[j]) / scale
    
    if x < z_unpaired:
        stochastic_traceback_stable(i, j-1, Z, unpaired_probs, paired_probs, beta, scale, structure, m)
        return

    cumulative_z = z_unpaired
    for k in range(i, j - m):
        term_left = Z[i, k-1] if k > i else 1.0
        z_paired = (term_left * np.exp(beta * paired_probs[k, j]) * Z[k+1, j-1]) / (scale**2)
        cumulative_z += z_paired
        
        if x < cumulative_z:
            structure[k], structure[j] = "(", ")"
            stochastic_traceback_stable(i, k-1, Z, unpaired_probs, paired_probs, beta, scale, structure, m)
            stochastic_traceback_stable(k+1, j-1, Z, unpaired_probs, paired_probs, beta, scale, structure, m)
            return

def sample_structures_stable(unpaired_probs, paired_probs, beta, num_samples=1000, m=3):
    """Generates samples using the stable partition function."""
    n = len(unpaired_probs)
    # Calculate Z and the scale factor used
    Z, scale = calculate_partition_function_stable(unpaired_probs, paired_probs, beta, m)
    
    samples = []
    for _ in range(num_samples):
        struct_list = ["."] * n
        stochastic_traceback_stable(0, n - 1, Z, unpaired_probs, paired_probs, beta, scale, struct_list, m)
        samples.append("".join(struct_list))
    return samples

# McCaskill algorithm using the ViennaRNA package

In [37]:
sequence = "GAGUAGUGGAACCAGGCUAUGUUUGUGACUCGCAGACCCU"

# 1. Create a 'fold_compound' for the sequence
md = RNA.md()  # create model details
md.uniq_ML = 1 # activate unique multibranch loop decomposition

fc = RNA.fold_compound(sequence, md)

# 2. Run the McCaskill algorithm (Forward pass)
# This calculates the partition function and returns (ensemble_energy)
(propensity, ensemble_energy) = fc.pf()

# 3. Get the base-pairing probability matrix (Backward pass)
# Returns a matrix where bpp[i][j] is the probability of i and j pairing
bpp_matrix = fc.bpp()

print(f"Ensemble Free Energy: {ensemble_energy:.2f} kcal/mol")
print(f"Probability of base 1 pairing with base 20: {bpp_matrix[1][20]:.4f}")

Ensemble Free Energy: -8.64 kcal/mol
Probability of base 1 pairing with base 20: 0.0005


In [3]:
print(propensity)
print(bpp_matrix)

..(((((........)))))(((((((...)))))))...
((0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.640343715432157e-06, 0.0, 0.0, 0.0, 0.0, 2.410004185660465e-05, 0.003714696211301517, 0.0, 0.0, 0.0, 5.9390471641559526e-05, 2.7592582865121717e-05, 0.0, 0.0004881023543541124, 0.0, 6.176627386760668e-05, 3.2218666960920665e-05, 9.152886501594468e-06, 0.0, 3.780650077304087e-08, 0.0, 0.0, 6.0960504916301e-08, 2.2490047656787905e-08, 0.013961111237572425, 0.0, 5.059886072987132e-05, 0.0, 0.0, 0.0, 3.499967121394817e-05, 0.0013273719742963573, 0.002900185449102967, 2.3317933484071404e-05), (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.421836596690229e-05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0237776806439977, 0.0, 0.0036530183426180442, 0.0, 5.019741582161185e-05, 1.2358685549285894e-05, 1.780211303

In [8]:
def boltzmann_sampling(fold_compound):
    """ 
    return a backtracked structure from a fold_compound where the partition function is already computed
    """ 
    return fold_compound.pbacktrack()

print(boltzmann_sampling(fc))

print("{}".format(fc.pbacktrack()))

...((((........)))).((((((.....))))))...
..(((((........)))))(((((((...)))))))...


# K-means clustering of structures

In [ ]:


import sys

# Create a fake distutils module to satisfy the import
try:
    import distutils.version
except ImportError:
    import packaging.version
    import types
    
    # Create a dummy module
    distutils = types.ModuleType("distutils")
    distutils.version = types.ModuleType("distutils.version")
    distutils.version.LooseVersion = packaging.version.Version
    sys.modules["distutils"] = distutils
    sys.modules["distutils.version"] = distutils.version

from sklearn_extra.cluster import KMedoids  # This should work now

## Distance functions

Hamming distance

In [40]:
def distance_hamming(str1, str2):
    """
    compute the hamming distance between two strings of equal length
    """
    assert(len(str1) == len(str2))
    return sum([str1[i] != str2[i] for i in range(len(str1))])


In [43]:
N = 300
samples = N*[None]
for i in range(N):
    samples[i] = fc.pbacktrack()


distance_matrix = np.array([[distance_hamming(samples[i], samples[j]) for i in range(N)] for j in range(N)])
kmedoids = KMedoids(n_clusters=3, metric='precomputed', random_state=42, init='k-medoids++')
clusters = kmedoids.fit(distance_matrix)

labels = kmedoids.labels_
medoid_indices = kmedoids.medoid_indices_

print(f"Cluster Assignments: {labels}")
print(f"Medoid Indices (The 'Centers'): {medoid_indices}")
print(f"Representative RNA for Cluster 0: {samples[medoid_indices[0]]}")


ValueError: could not convert string to float: '..(((((........)))))(((((((...)))))))...'